# Notebook 03: Geospatial Feature Engineering & Multicollinearity Audit (Phase 5)
**Objectives:**
1. Engineer physically interpretable Mineral Systems predictors (fluid pathways, redox chemical traps, pathfinder log-ratios, local magnetic residuals, and topographic wetness indices).
2. Audit multicollinearity via **Variance Inflation Factors (VIF)** so linear baseline coefficients remain stable.
3. Persist the feature-engineered GeoPackage for modeling.

In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.acquisition import generate_zambia_prospectivity_benchmark, load_config
from src.data.cleaning import validate_and_standardize_geodata
from src.features.spatial_features import compute_vif_table, engineer_geospatial_features

config = load_config(PROJECT_ROOT / "configs" / "config.yaml")
raw_gdf, faults_gdf = generate_zambia_prospectivity_benchmark(config)
clean_gdf, _ = validate_and_standardize_geodata(raw_gdf, target_crs=config["crs"]["projected_crs"])

feat_gdf, feature_cols = engineer_geospatial_features(clean_gdf, faults_gdf)
print(f"Engineered {len(feature_cols)} features:")
for col in feature_cols:
    print(f" - {col}")

Engineered 21 features:
 - dist_to_fault_m
 - dist_to_redox_contact_m
 - fault_proximity_exp
 - fault_density_10km
 - redox_trap_proximity_exp
 - structural_redox_interaction
 - log10_cu_ppm
 - log10_co_ppm
 - local_cu_halo_log10
 - cu_co_log_ratio
 - mag_anomaly_nt
 - mag_residual_nt
 - bouguer_gravity_mgal
 - radiometric_k_th_ratio
 - elevation_m
 - slope_deg
 - topographic_wetness_proxy
 - lith_Kundelungu_Glaciogenic_Siltstone
 - lith_Lower_Roan_Ore_Shale_Arkose
 - lith_Mwashia_Carbonaceous_Shale
 - lith_Upper_Roan_Dolomite


## 1. Variance Inflation Factor (VIF) Multicollinearity Check
Notice that including `log10_cu_ppm`, `log10_co_ppm`, and `cu_co_log_ratio` simultaneously creates exact linear collinearity ($\log_{10}(\text{Cu}) - \log_{10}(\text{Co}) = \log_{10}(\text{Cu}/\text{Co})$). We inspect VIF and drop exact linear combinations before fitting Logistic Regression.

In [2]:
modeling_features = [c for c in feature_cols if c != "cu_co_log_ratio"]
vif_df = compute_vif_table(feat_gdf, modeling_features)
display(vif_df)

# Save engineered dataset
out_path = PROJECT_ROOT / config["data_paths"]["processed_dir"] / "zambia_engineered_features.gpkg"
out_path.parent.mkdir(parents=True, exist_ok=True)
feat_gdf.to_file(out_path, driver="GPKG")
print(f"Saved engineered feature layer to {out_path}")

,feature,VIF
0,lith_Kundelungu_Glaciogenic_Siltstone,43.69
1,mag_anomaly_nt,20.51
2,lith_Mwashia_Carbonaceous_Shale,14.24
3,bouguer_gravity_mgal,11.82
4,redox_trap_proximity_exp,8.75
5,lith_Upper_Roan_Dolomite,7.62
6,slope_deg,6.42
7,lith_Lower_Roan_Ore_Shale_Arkose,6.29
8,topographic_wetness_proxy,5.77
9,fault_proximity_exp,5.73


Saved engineered feature layer to C:\Users\JSONGA5\Desktop\kobold\data\processed\zambia_engineered_features.gpkg
